In [3]:
%load_ext autoreload
%autoreload 2
import pandas as pd

from absolute_pipeline.ingestion import (
    prepare_volume_records,
    load_volume_records
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
import hashlib
import json

volume = {
    "id": 160294,
    "name": "Absolute Batman",
    "count_of_issues": 21
}

In [5]:
serialized = json.dumps(volume, sort_keys=True)

encoded = serialized.encode("utf-8")

record_hash = hashlib.sha256(encoded).hexdigest()

print(record_hash)

7f7a988e4afddc5fea54dc614e96d37e5eb0eb778940e821a2f84fa34991b322


In [8]:
serialized = json.dumps(volume, sort_keys=True)

encoded = serialized.encode("utf-8")

record_hash = hashlib.sha256(encoded).hexdigest()

print(record_hash)

dd4cf4b892b91050a585248b52358a58a9bf13935e4e722be2c645453a256859


In [9]:
from google.cloud import bigquery

client = bigquery.Client()

In [10]:
# Create Control table

schema = [
    bigquery.SchemaField(
        "volume_id",
        "INT64",
        mode="REQUIRED"
    ),
    bigquery.SchemaField(
        "issue_count",
        "INT64",
        mode="NULLABLE"
    ),
    bigquery.SchemaField(
        "last_issue_id",
        "INT64",
        mode="NULLABLE"
    ),
     bigquery.SchemaField(
        "source_updated_at",
        "TIMESTAMP",
        mode="NULLABLE"
    ),
     bigquery.SchemaField(
        "last_checked_at",
        "TIMESTAMP",
        mode="REQUIRED"
    ),
     bigquery.SchemaField(
        "last_fetched_at",
        "TIMESTAMP",
        mode="REQUIRED"
    ),
     bigquery.SchemaField(
        "record_hash",
        "STRING",
        mode="REQUIRED"
    ),
    bigquery.SchemaField(
        "is_tracked",
        "BOOL",
        mode="REQUIRED"
    ),
      bigquery.SchemaField(
        "run_id",
        "STRING",
        mode="REQUIRED"
    ),

]

table_id = f"{client.project}.control.volumes"

table = bigquery.Table(
    table_id,
    schema=schema
)

client.create_table(
    table,
    exists_ok=True
)

Table(TableReference(DatasetReference('dc-absolute', 'control'), 'volumes'))

In [11]:
def calculate_record_hash(record: dict) -> str:
    # Serialize the record
    serialized = json.dumps(record, sort_keys=True)

    # Encode the serialized record
    encoded = serialized.encode("utf-8")

    # Calculate the SHA-256 hash
    record_hash = hashlib.sha256(encoded).hexdigest()

    # Return the hash
    return record_hash

In [1]:
from absolute_pipeline.utils import calculate_record_hash 

volume_a = {
    "id": 160294,
    "name": "Absolute Batman",
    "count_of_issues": 25
}

volume_b = {
    "name": "Absolute Batman",
    "count_of_issues": 25,
    "id": 160294
}

volume_c = volume_a.copy()
volume_c["count_of_issues"] = 26

hash_a = calculate_record_hash(volume_a)
hash_b = calculate_record_hash(volume_b)
print(hash_a)
print(hash_b)

714ecbd239b914a47f018936066b5e7d992f1101e7a9a5c7463341a49d300d23
714ecbd239b914a47f018936066b5e7d992f1101e7a9a5c7463341a49d300d23


In [14]:
assert calculate_record_hash(volume_a) == calculate_record_hash(volume_b)
assert calculate_record_hash(volume_a) != calculate_record_hash(volume_c)
assert len(calculate_record_hash(volume_a)) == 64

In [19]:
query = """ SELECT
     volume_id,
    payload,
    -- Extract name
    JSON_VALUE(payload, '$.name') AS name,
    -- Extract issue_count
    SAFE_CAST(JSON_VALUE(payload, '$.count_of_issues') AS INT64) AS issue_count,
    -- Extract last_issue_id
        SAFE_CAST(JSON_VALUE(payload, '$.last_issue.id') AS INT64) AS last_issue_id,
    source_updated_at,
    extracted_at,
    run_id
FROM bronze.volumes
 QUALIFY ROW_NUMBER() OVER (
        PARTITION BY volume_id
        ORDER BY extracted_at DESC, run_id DESC
    ) = 1;"""

query_job = client.query(query)

results = query_job.result()


In [20]:
bronze_records = [dict(row) for row in results]
bronze_records[0]

{'volume_id': 116693,
 'payload': {'aliases': None,
  'api_detail_url': 'https://comicvine.gamespot.com/api/volume/4050-116693/',
  'count_of_issues': 1,
  'date_added': '2019-01-23 04:15:10',
  'date_last_updated': '2019-01-23 06:11:04',
  'deck': ' ',
  'description': '<p>Hardcover Absolute Edition collecting <a href="/flashpoint/4050-39997/" data-ref-id="4050-39997">Flashpoint</a>.</p>',
  'first_issue': {'api_detail_url': 'https://comicvine.gamespot.com/api/issue/4000-698575/',
   'id': 698575,
   'issue_number': '1',
   'name': 'HC'},
  'id': 116693,
  'image': {'icon_url': 'https://comicvine.gamespot.com/a/uploads/square_avatar/6/67663/6771132-01.jpg',
   'image_tags': 'All Images',
   'medium_url': 'https://comicvine.gamespot.com/a/uploads/scale_medium/6/67663/6771132-01.jpg',
   'original_url': 'https://comicvine.gamespot.com/a/uploads/original/6/67663/6771132-01.jpg',
   'screen_large_url': 'https://comicvine.gamespot.com/a/uploads/screen_kubrick/6/67663/6771132-01.jpg',
   's

In [21]:
for key, value in bronze_records[0].items():
    print(f"{key}: {type(value).__name__}")

volume_id: int
payload: dict
name: str
issue_count: int
last_issue_id: int
source_updated_at: NoneType
extracted_at: datetime
run_id: str


In [22]:
assert len(bronze_records) == len({
    record["volume_id"]
    for record in bronze_records
})

In [30]:
def prepare_control_records(volumes: list[dict],tracked_volume_ids: set[int]) -> list[dict]:

    records = []

    for volume in volumes:

        # Prepare payload
        payload = volume["payload"]

        if isinstance(payload, str):
            payload = json.loads(payload)

        # Calculate record hash
        record_hash = calculate_record_hash(payload)

        # Prepare timestamps
        extracted_at = volume["extracted_at"].isoformat()

        source_updated_at = (
            volume["source_updated_at"].isoformat()
            if volume["source_updated_at"] is not None
            else None
        )

        # Build control record
        record = {
            "volume_id": volume["volume_id"],
            "issue_count": volume["issue_count"],
            "last_issue_id": volume["last_issue_id"],
            "source_updated_at": source_updated_at,
            "last_checked_at": extracted_at,
            "last_fetched_at": extracted_at,
            "record_hash": record_hash,
            "is_tracked": volume["volume_id"] in tracked_volume_ids,
            "run_id": volume["run_id"]
        }

        records.append(record)

    return records

In [31]:
payloads = [
    json.loads(record["payload"])
    if isinstance(record["payload"], str)
    else record["payload"]
    for record in bronze_records
]



raw_df = pd.json_normalize(payloads, sep="_")
raw_df["start_year"] = pd.to_numeric(
    raw_df["start_year"],
    errors="coerce"
)

filtered_df = raw_df[
    (raw_df["publisher_name"] == "DC Comics")
    & (raw_df["start_year"] >= 2024)
    & (~raw_df["name"].str.contains("Absolute Power", na=False))
].copy()


tracked_volume_ids = set(filtered_df["id"].astype(int))

In [32]:
control_records = prepare_control_records(
    volumes=bronze_records,
    tracked_volume_ids=tracked_volume_ids
)

In [33]:
control_records[0]

{'volume_id': 116693,
 'issue_count': 1,
 'last_issue_id': 698575,
 'source_updated_at': None,
 'last_checked_at': '2026-11-26T00:01:27.566074+00:00',
 'last_fetched_at': '2026-11-26T00:01:27.566074+00:00',
 'record_hash': '82acdbc3da933c4b178f43364231e435b0d17b4766f340b020be1d9869b3d306',
 'is_tracked': False,
 'run_id': 'f63684d4-2fc5-45ca-9093-59eb9bb69044'}

In [34]:
# Validate record count
assert len(control_records) == len(bronze_records)

# Validate unique volume IDs
assert len(control_records) == len({
    record["volume_id"]
    for record in control_records
})

# Validate SHA-256 hashes
assert all(
    len(record["record_hash"]) == 64
    for record in control_records
)

# Validate JSON serialization
json.dumps(control_records)

print("All validations passed")

All validations passed


In [35]:
control_df = pd.DataFrame(control_records)

control_df["is_tracked"].value_counts()

is_tracked
False    151
True      23
Name: count, dtype: int64

In [36]:
control_table_id = f"{client.project}.control.volumes"

control_table = client.get_table(control_table_id)

control_table.schema

[SchemaField('volume_id', 'INTEGER', 'REQUIRED', None, None, (), None, None, None, None),
 SchemaField('issue_count', 'INTEGER', 'NULLABLE', None, None, (), None, None, None, None),
 SchemaField('last_issue_id', 'INTEGER', 'NULLABLE', None, None, (), None, None, None, None),
 SchemaField('source_updated_at', 'TIMESTAMP', 'NULLABLE', None, None, (), None, None, None, None),
 SchemaField('last_checked_at', 'TIMESTAMP', 'REQUIRED', None, None, (), None, None, None, None),
 SchemaField('last_fetched_at', 'TIMESTAMP', 'REQUIRED', None, None, (), None, None, None, None),
 SchemaField('record_hash', 'STRING', 'REQUIRED', None, None, (), None, None, None, None),
 SchemaField('is_tracked', 'BOOLEAN', 'REQUIRED', None, None, (), None, None, None, None),
 SchemaField('run_id', 'STRING', 'REQUIRED', None, None, (), None, None, None, None)]

In [45]:
staging_table_id = f"{client.project}.control.volumes_staging"

staging_table = bigquery.Table(
    staging_table_id, 
   control_table.schema
)

client.create_table(
    staging_table,
    exists_ok=True
)

Table(TableReference(DatasetReference('dc-absolute', 'control'), 'volumes_staging'))

In [46]:
def load_staging_records(client: bigquery.Client, records: list[dict], table_id: str):
    # Configure the load job
        job_config = bigquery.LoadJobConfig(
            schema=client.get_table(table_id).schema,
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
        )
    
        # Submit the load job
        job = client.load_table_from_json(json_rows=records, destination=table_id, job_config=job_config)
    
        # Wait for completion
        job.result()
    
        # Print the number of loaded rows
        print(job.job_id)
        print(job.state)
        print(job.output_rows)
        print(job.errors)
        return job

In [47]:
job = load_staging_records(
    client=client,
    records=control_records,
    table_id=staging_table_id
)

5dbbeef9-0be4-4193-9e87-ec426d0b041d
DONE
174
None


In [40]:
assert job.output_rows == len(control_records)

In [48]:
merge_query = """
MERGE control.volumes AS T
USING control.volumes_staging AS S
ON T.volume_id = S.volume_id

WHEN NOT MATCHED THEN
    INSERT (
        volume_id,
        issue_count,
        last_issue_id,
        source_updated_at,
        last_checked_at,
        last_fetched_at,
        record_hash,
        is_tracked,
        run_id

    )
    VALUES (
        S.volume_id,
        S.issue_count,
        S.last_issue_id,
        S.source_updated_at,
        S.last_checked_at,
        S.last_fetched_at,
        S.record_hash,
        S.is_tracked,
        S.run_id
    );
"""

merge_job = client.query(merge_query)
merge_job.result()

print(f"Affected rows: {merge_job.num_dml_affected_rows}")

Affected rows: 0


In [49]:
def get_control_state( client: bigquery.Client, table_id: str) -> dict[int, dict]:

    # Build the SQL query
    query = f"""
        SELECT *
        FROM `{table_id}`
    """

    # Execute the query
    query_job = client.query(query)
    results = query_job.result()

    # Build a dictionary indexed by volume_id
    control_state = { row["volume_id"]: dict(row) for row in results
    }

    return control_state

In [50]:
control_state = get_control_state(
    client=client,
    table_id=f"{client.project}.control.volumes"
)

print(f"Total volumes: {len(control_state)}")

first_volume_id = next(iter(control_state))

print(control_state[first_volume_id])

Total volumes: 174
{'volume_id': 128280, 'issue_count': 1, 'last_issue_id': 771706, 'source_updated_at': None, 'last_checked_at': datetime.datetime(2026, 11, 26, 0, 1, 27, 566074, tzinfo=datetime.timezone.utc), 'last_fetched_at': datetime.datetime(2026, 11, 26, 0, 1, 27, 566074, tzinfo=datetime.timezone.utc), 'record_hash': 'd94af43203e5afb4f7d208376639194e9211259275c97ea8553afaf5c56a8d9f', 'is_tracked': False, 'run_id': 'f63684d4-2fc5-45ca-9093-59eb9bb69044'}


In [51]:
tracked_volumes = {
    volume_id: volume
    for volume_id, volume in control_state.items()
    if volume["is_tracked"]
}

print(f"Tracked volumes: {len(tracked_volumes)}")

Tracked volumes: 23


In [52]:
mock_volume_a = {
    "id": 100,
    "name": "Absolute Batman",
    "count_of_issues": 25
}

mock_volume_b = {
    "id": 200,
    "name": "Absolute Superman",
    "count_of_issues": 20
}

mock_volume_c = {
    "id": 300,
    "name": "Absolute Wonder Woman",
    "count_of_issues": 15
}

mock_control_state = {
    100: {
        "volume_id": 100,
        "issue_count": 25,
        "record_hash": calculate_record_hash(mock_volume_a),
        "is_tracked": True
    },
    200: {
        "volume_id": 200,
        "issue_count": 20,
        "record_hash": calculate_record_hash(mock_volume_b),
        "is_tracked": True
    }
}

In [55]:
mock_control_state.get(200)

{'volume_id': 200,
 'issue_count': 20,
 'record_hash': 'c9153d68354992e7dec197acd4b3b108737da1088ca0b9e0101643d53449a349',
 'is_tracked': True}

In [53]:
mock_volume_b_updated = mock_volume_b.copy()

mock_volume_b_updated["count_of_issues"] = 21

In [54]:
mock_incoming_volumes = [
    mock_volume_a,
    mock_volume_b_updated,
    mock_volume_c
]

In [ ]:
def detect_volume_changes(volumes: list[dict], control_state: dict[int, dict]) -> dict[str, list[dict]]:

    changes = {
        "new": [],
        "updated": [],
        "unchanged": []
    }

    for volume in volumes:

        # Get volume ID
        volume_id= volume["id"]

        # Calculate incoming record hash
        volume_hash = calculate_record_hash(volume)

        # Get previous state
        previous_state = control_state.get(volume_id)

        # Classify volume
        if previous_state is None:
            changes["new"].append(volume)

        elif volume_hash != previous_state["record_hash"]:
            changes["updated"].append(volume)

        else:
            changes["unchanged"].append(volume)

    return changes

In [60]:
changes = detect_volume_changes(
    volumes=mock_incoming_volumes,
    control_state=mock_control_state
)

changes


{'new': [{'id': 300, 'name': 'Absolute Wonder Woman', 'count_of_issues': 15}],
 'updated': [{'id': 200, 'name': 'Absolute Superman', 'count_of_issues': 21}],
 'unchanged': [{'id': 100, 'name': 'Absolute Batman', 'count_of_issues': 25}]}

In [ ]:
assert changes["new"][0]["id"] == 300
assert changes["updated"][0]["id"] == 200
assert changes["unchanged"][0]["id"] == 100



In [62]:
changes = detect_volume_changes(
    volumes=mock_incoming_volumes,
    control_state=mock_control_state
)

for status, volumes in changes.items():
    print(f"{status.upper()}: {len(volumes)}")

NEW: 1
UPDATED: 1
UNCHANGED: 1


In [63]:
assert [v["id"] for v in changes["new"]] == [300]
assert [v["id"] for v in changes["updated"]] == [200]
assert [v["id"] for v in changes["unchanged"]] == [100]

print("All validations passed")

All validations passed


In [64]:
assert detect_volume_changes([], mock_control_state) == {
    "new": [],
    "updated": [],
    "unchanged": []
}

In [65]:
changes = detect_volume_changes(
    mock_incoming_volumes,
    {}
)

assert len(changes["new"]) == 3
assert len(changes["updated"]) == 0
assert len(changes["unchanged"]) == 0

In [66]:
changes = detect_volume_changes(
    [mock_volume_a, mock_volume_b],
    mock_control_state
)

assert len(changes["new"]) == 0
assert len(changes["updated"]) == 0
assert len(changes["unchanged"]) == 2